In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os


dataset_dir = '/content/drive/Shareddrives/STAI_Project/datasets/image'

output_base = '/content/drive/Shareddrives/STAI_Project/datasets/csv/splits_csv'

os.makedirs(output_base, exist_ok=True)


make array of all image paths and its labels


In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

image_paths = []
labels = []

for label_name in tqdm(os.listdir(dataset_dir)):
    label_path = os.path.join(dataset_dir, label_name)
    if os.path.isdir(label_path):
        for img_name in os.listdir(label_path):
            if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                image_paths.append(os.path.join(label_path, img_name))
                labels.append(label_name)

image_paths = np.array(image_paths)
labels = np.array(labels)

print(f" Total images: {len(image_paths)}")
print(f" Total classes: {len(np.unique(labels))}")


100%|██████████| 64/64 [00:00<00:00, 97.72it/s]

 Total images: 14377
 Total classes: 64


Apply stratified k-fold cross validation with 5 folds on the dataset.and make train validation test split and make csv file for each.

In [ ]:
from sklearn.model_selection import StratifiedKFold, train_test_split

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold = 1
for train_val_idx, test_idx in skf.split(image_paths, labels):
    X_train_val, X_test = image_paths[train_val_idx], image_paths[test_idx]
    y_train_val, y_test = labels[train_val_idx], labels[test_idx]


    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=0.2, stratify=y_train_val, random_state=42
    )


    fold_dir = os.path.join(output_base, f'fold_{fold}')
    os.makedirs(fold_dir, exist_ok=True)

    pd.DataFrame({'image_path': X_train, 'label': y_train}).to_csv(os.path.join(fold_dir, 'train.csv'), index=False)
    pd.DataFrame({'image_path': X_val, 'label': y_val}).to_csv(os.path.join(fold_dir, 'val.csv'), index=False)
    pd.DataFrame({'image_path': X_test, 'label': y_test}).to_csv(os.path.join(fold_dir, 'test.csv'), index=False)

    print(f" Fold {fold} saved in: {fold_dir}")
    fold += 1




 Fold 1 saved in: /content/drive/Shareddrives/STAI_Project/datasets/csv/splits_csv/fold_1
 Fold 2 saved in: /content/drive/Shareddrives/STAI_Project/datasets/csv/splits_csv/fold_2
 Fold 3 saved in: /content/drive/Shareddrives/STAI_Project/datasets/csv/splits_csv/fold_3
 Fold 4 saved in: /content/drive/Shareddrives/STAI_Project/datasets/csv/splits_csv/fold_4
 Fold 5 saved in: /content/drive/Shareddrives/STAI_Project/datasets/csv/splits_csv/fold_5


show length of each fold

In [ ]:
for i in range(1, 6):
    print(f"\nFold {i}")
    for split_name in ['train', 'val', 'test']:
        df = pd.read_csv(os.path.join(output_base, f'fold_{i}', f'{split_name}.csv'))
        counts = df['label'].value_counts().sort_index()
        print(f"{split_name}: {len(df)} images across {len(counts)} classes")



Fold 1
train: 9200 images across 64 classes
val: 2301 images across 64 classes
test: 2876 images across 64 classes

Fold 2
train: 9200 images across 64 classes
val: 2301 images across 64 classes
test: 2876 images across 64 classes

Fold 3
train: 9201 images across 64 classes
val: 2301 images across 64 classes
test: 2875 images across 64 classes

Fold 4
train: 9201 images across 64 classes
val: 2301 images across 64 classes
test: 2875 images across 64 classes

Fold 5
train: 9201 images across 64 classes
val: 2301 images across 64 classes
test: 2875 images across 64 classes
